In [2]:
import pandas as pd
import plotly.express as px

# Load source files
data = pd.read_csv("data.csv")
country_info = pd.read_csv("country_info.csv")

# Tidy column names and drop unnamed columns
data.columns = data.columns.str.strip()
country_info.columns = country_info.columns.str.strip()
data = data.loc[:, data.columns.notna()]
data = data.loc[:, ~data.columns.str.contains(r'^Unnamed', na=False)]

pop_col = "Population ages 65 and above (% of total population)"

# Ensure numeric population share
data[pop_col] = pd.to_numeric(data[pop_col], errors="coerce")

# Merge on Country Code to avoid name mismatches
merged = data.merge(
    country_info[["Country Code", "IncomeGroup"]],
    on="Country Code",
    how="left",
    validate="many_to_one",
    indicator=True,
)

missing_countries = merged.loc[merged["_merge"] == "left_only", ["Country Name", "Country Code"]].reset_index(drop=True)
merged["IncomeGroup"] = merged["IncomeGroup"].fillna("Not classified")

value_min = merged[pop_col].min()
value_max = merged[pop_col].max()

print("Integrity check")
print(f"Total rows processed: {len(merged)}")
print(f"Range of population ages 65+ (%): {value_min:.2f} to {value_max:.2f}")
top5 = merged.sort_values(pop_col, ascending=False).head(5)[["Country Name", pop_col]]
print("Top 5 oldest countries:")
print(top5.to_string(index=False))

if not missing_countries.empty:
    print("Countries in data.csv missing IncomeGroup mapping (by Country Code):")
    print(missing_countries.to_string(index=False))
else:
    print("No countries missing IncomeGroup mapping.")

fig = px.choropleth(
    merged,
    locations="Country Code",
    color=pop_col,
    hover_name="Country Name",
    hover_data={"Country Name": True, pop_col: ":.2f", "IncomeGroup": True},
    color_continuous_scale="Viridis",
    range_color=(value_min, value_max),
    projection="natural earth",
    title="Population Ages 65+ (%) by Country",
    labels={pop_col: "Population ages 65+ (%)"},
)
fig.update_layout(coloraxis_colorbar_title="Population ages 65+ (%)")
fig.show()

Integrity check
Total rows processed: 266
Range of population ages 65+ (%): 1.57 to 36.36
Top 5 oldest countries:
Country Name  Population ages 65 and above (% of total population)
      Monaco                                             36.357219
       Japan                                             29.561810
 Puerto Rico                                             24.243724
       Italy                                             24.218783
    Portugal                                             24.111511
Countries in data.csv missing IncomeGroup mapping (by Country Code):
  Country Name Country Code
Not classified          INX


# Verification Report By AI

- **Evolution Loop 1 (Mapping Integrity):** Kept primary key on Country Code; added ISO-3 fuzzy fallback for low coverage cases (Turkey/Türkiye, Slovakia, South Korea, etc.) and now report coverage plus unmapped rows.

- **Evolution Loop 2 (Visual Distortion Check):** Auto-detect skew; switches to log-scaled colors when high skew is detected while keeping a continuous Viridis bar with readable tick labels in percent.

- **Evolution Loop 3 (Data-Visual Alignment):** Prints the top 10 values actually rendered on the map and highlights any rows still lacking IncomeGroup, enabling quick cross-check against the CSV.

- Run Cell 2 to execute the improved pipeline and view both the map and printed verification outputs.


In [4]:
import numpy as np

from pathlib import Path



import pycountry

import pandas as pd

import plotly.express as px



# Load source files

data = pd.read_csv("data.csv")

country_info = pd.read_csv("country_info.csv")



# Tidy column names and drop unnamed columns

data.columns = data.columns.str.strip()

country_info.columns = country_info.columns.str.strip()

data = data.loc[:, data.columns.notna()]

data = data.loc[:, ~data.columns.str.contains(r'^Unnamed', na=False)]

data["Country Code"] = data["Country Code"].str.strip()



pop_col = "Population ages 65 and above (% of total population)"



# Ensure numeric population share

data[pop_col] = pd.to_numeric(data[pop_col], errors="coerce")



# Helper: ISO3 lookup for fuzzy country names (used only if merge coverage is low)

manual_iso_overrides = {

    "Turkey": "TUR",

    "Turkiye": "TUR",

    "Türkiye": "TUR",

    "Slovakia": "SVK",

    "Slovak Republic": "SVK",

    "South Korea": "KOR",

    "Republic of Korea": "KOR",

    "Korea, Rep.": "KOR",

    "North Korea": "PRK",

    "Democratic People's Republic of Korea": "PRK",

    "Congo, Dem. Rep.": "COD",

    "Congo, Rep.": "COG",

    "Bahamas": "BHS",

}



def iso3_from_name(name: str) -> str | None:

    if pd.isna(name):

        return None

    if name in manual_iso_overrides:

        return manual_iso_overrides[name]

    try:

        return pycountry.countries.search_fuzzy(name)[0].alpha_3

    except Exception:

        return None



# First-pass merge on Country Code (strict key)

merged = data.merge(

    country_info[["Country Code", "IncomeGroup"]],

    on="Country Code",

    how="left",

    validate="many_to_one",

    indicator=True,

)



coverage = 1 - merged["IncomeGroup"].isna().mean()



# If coverage drops below 95%, retry with ISO-3 lookup on unresolved rows

if coverage < 0.95:

    data_fuzzy = data.copy()

    needs_help = merged["IncomeGroup"].isna()

    data_fuzzy["merge_code"] = data_fuzzy["Country Code"]

    data_fuzzy.loc[needs_help, "merge_code"] = data_fuzzy.loc[needs_help, "Country Name"].apply(iso3_from_name)

    merged = data_fuzzy.merge(

        country_info[["Country Code", "IncomeGroup"]],

        left_on="merge_code",

        right_on="Country Code",

        how="left",

        validate="many_to_one",

        indicator=True,

    )

    merged["Country Code"] = merged["merge_code"]

    merged.drop(columns=["merge_code"], inplace=True)

    coverage = 1 - merged["IncomeGroup"].isna().mean()



merged["IncomeGroup"] = merged["IncomeGroup"].fillna("Not classified")



# Identify unmatched rows

missing_countries = merged.loc[

    merged["IncomeGroup"] == "Not classified",

    ["Country Name", "Country Code"],

].reset_index(drop=True)



# Range stats

value_min = merged[pop_col].min()

value_max = merged[pop_col].max()



# Decide color scaling to reduce distortion

color_series = merged[pop_col]

use_log_scale = (color_series.max() / max(color_series.median(), 1e-6)) > 2.5

if use_log_scale:

    merged["color_value"] = np.log1p(color_series)

    color_label = "Population ages 65+ (%) - log scale"

else:

    merged["color_value"] = color_series

    color_label = "Population ages 65+ (%)"



ticks = np.linspace(value_min, value_max, num=6)

colorbar_params = {

    "title": color_label,

    "tickvals": np.log1p(ticks) if use_log_scale else ticks,

    "ticktext": [f"{t:.1f}%" for t in ticks],

}



# Choropleth

fig = px.choropleth(

    merged,

    locations="Country Code",

    color="color_value",

    hover_name="Country Name",

    hover_data={"Country Name": True, pop_col: ":.2f", "IncomeGroup": True},

    color_continuous_scale="Viridis",

    range_color=(merged["color_value"].min(), merged["color_value"].max()),

    projection="natural earth",

    title="Population Ages 65+ (%) by Country",

    labels={"color_value": color_label},

)

fig.update_layout(

    coloraxis_colorbar=colorbar_params,

    margin=dict(l=20, r=20, t=70, b=20),

    paper_bgcolor="white",

    geo=dict(showframe=False, showcoastlines=False, bgcolor="white"),

)



# Save the raw interactive HTML version

html_path = Path("world_aging_map_interactive.html")

fig.write_html(html_path, full_html=True, include_plotlyjs="cdn")



# Save a self-contained shareable HTML version

share_html_path = Path("world_aging_map_share.html")

plot_html = fig.to_html(

    full_html=False,

    include_plotlyjs=True,

    config={

        "responsive": True,

        "displaylogo": False,

        "toImageButtonOptions": {

            "format": "png",

            "filename": "world_aging_map",

            "scale": 2,

        },

    },

)



share_page = f"""<!DOCTYPE html>

<html lang=\"en\">

<head>

  <meta charset=\"utf-8\">

  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">

  <title>Population Ages 65+ (%) by Country</title>

  <style>

    body {{

      margin: 0;

      font-family: Arial, Helvetica, sans-serif;

      background: #f5f7fb;

      color: #1f2937;

    }}

    .page {{

      max-width: 1200px;

      margin: 0 auto;

      padding: 32px 20px 48px;

    }}

    .card {{

      background: white;

      border-radius: 16px;

      box-shadow: 0 10px 30px rgba(15, 23, 42, 0.08);

      padding: 24px;

    }}

    h1 {{

      margin: 0 0 8px;

      font-size: 2rem;

    }}

    p {{

      line-height: 1.6;

    }}

    .meta {{

      color: #4b5563;

      margin-bottom: 18px;

    }}

    .note {{

      margin-top: 18px;

      font-size: 0.95rem;

      color: #374151;

    }}

  </style>

</head>

<body>

  <div class=\"page\">

    <div class=\"card\">

      <h1>Population Ages 65+ (%) by Country</h1>

      <p class=\"meta\">Interactive choropleth map built from data.csv and country_info.csv for SEEM2460 Assignment 2.</p>

      <p>This interactive figure shows the geographic distribution of the population aged 65 and above. The map color encodes ageing percentage, while IncomeGroup is provided through the hover information to support interpretation of the spatial pattern.</p>

      {plot_html}

      <p class=\"note\"><strong>Reading note:</strong> Hover over a country to view its Country Name, ageing percentage, and IncomeGroup. </p>

    </div>

  </div>

</body>

</html>

"""



share_html_path.write_text(share_page, encoding="utf-8")



# Integrity check

print("Integrity check (after mapping evolution)")

print(f"IncomeGroup coverage: {coverage:.1%}")

print(f"Total rows processed: {len(merged)}")

print(f"Range of population ages 65+ (%): {value_min:.2f} to {value_max:.2f}")

top5 = merged.sort_values(pop_col, ascending=False).head(5)[["Country Name", pop_col]]

print("Top 5 oldest countries:")

print(top5.to_string(index=False))



if not missing_countries.empty:

    print("Countries missing IncomeGroup mapping (by Country Code):")

    print(missing_countries.to_string(index=False))

else:

    print("No countries missing IncomeGroup mapping.")



# Verification table: top 10 values used in choropleth

top10 = merged.sort_values(pop_col, ascending=False).head(10)[[

    "Country Name",

    "Country Code",

    pop_col,

    "IncomeGroup",

]]

print("\nTop 10 values rendered on map:")

print(top10.to_string(index=False))

print(f"\nInteractive HTML saved to: {html_path.resolve()}")

print(f"Share-ready HTML saved to: {share_html_path.resolve()}")

print("Upload the share-ready HTML file to a static hosting service to get a public link.")



fig.show()


Integrity check (after mapping evolution)
IncomeGroup coverage: 81.2%
Total rows processed: 266
Range of population ages 65+ (%): 1.57 to 36.36
Top 5 oldest countries:
Country Name  Population ages 65 and above (% of total population)
      Monaco                                             36.357219
       Japan                                             29.561810
 Puerto Rico                                             24.243724
       Italy                                             24.218783
    Portugal                                             24.111511
Countries missing IncomeGroup mapping (by Country Code):
                                        Country Name Country Code
                         Africa Eastern and Southern         None
                          Africa Western and Central         None
                                          Arab World         None
                      Central Europe and the Baltics         None
                              Caribbean sma